# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [22]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests


# Import API key
from config import geoapify_key 

In [23]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head(10)

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,papatowai,-46.5619,169.4708,51.69,69,29,12.26,NZ,1787104408
1,1,karasburg,-28.0167,18.7500,62.64,17,0,6.11,NaN,1787104409
2,2,port-aux-francais,-49.3500,70.2167,31.93,93,100,19.28,TF,1787104413
3,3,olonkinbyen,70.9221,-8.7187,41.07,82,100,10.56,SJ,1787104414
4,4,ushuaia,-54.8000,-68.3000,38.86,81,94,21.85,AR,1787104416
5,5,port mathurin,-19.6833,63.4167,71.69,78,30,17.27,MU,1787104417
6,6,ribeira grande,38.5167,-28.7000,74.77,88,3,7.00,PT,1787104419
7,7,hitzacker,53.1525,11.0442,62.40,92,100,9.28,DE,1787104420
8,8,dunedin,-45.8742,170.5036,53.62,70,50,11.01,NZ,1787104314
9,9,aasiaat,68.7098,-52.8699,38.52,98,100,5.39,GL,1787104423


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [24]:
#%%capture --no-display

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    x= 'Lng', y='Lat',
    geo= True, tiles='OSM',
    size='Humidity',
    color='City',
    cmap='Category10',
    legend='bottom_right',
    alpha=0.5,
    hover_cols=['City','Humidity']
)

#Display the map plot
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [25]:
filtered_df = city_data_df[city_data_df['Humidity'] < 50]

city_data_df = pd.DataFrame(city_data_df)

#Drop null values
updated_df = filtered_df.dropna()

#Display sample dateframe 
updated_df.head(10)

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
21,21,glen innes,-29.7333,151.7333,58.10,47,4,5.97,AU,1787104442
22,22,anaconda,46.1285,-112.9423,72.25,33,13,6.49,US,1787104444
43,43,tindouf,27.6711,-8.1474,95.45,15,69,10.36,DZ,1787104480
66,66,kununurra,-15.7667,128.7333,85.98,45,8,6.91,AU,1787104517
78,78,korla,41.7597,86.1469,81.70,26,100,14.70,CN,1787104535
84,84,gurupi,-11.7292,-49.0686,81.46,31,43,1.36,BR,1787104544
96,96,luena,-11.7833,19.9167,51.66,34,0,4.34,AO,1787104562
100,100,whitehorse,60.7161,-135.0538,66.97,37,94,2.30,CA,1787104410
103,103,najafabad,32.6344,51.3668,79.59,15,0,2.35,IR,1787104573
110,110,grantsville,40.5999,-112.4644,88.56,23,3,5.75,US,1787104584


### Step 3: Create a new DataFrame called `hotel_df`.

In [26]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df = updated_df[['City', 'Country', 'Lat','Lng', 'Humidity']].copy()

hotel_df = pd.DataFrame(hotel_df)

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df['Hotel name'] = " "

# Display sample data
hotel_df.head(10)

,City,Country,Lat,Lng,Humidity,Hotel name
21,glen innes,AU,-29.7333,151.7333,47,
22,anaconda,US,46.1285,-112.9423,33,
43,tindouf,DZ,27.6711,-8.1474,15,
66,kununurra,AU,-15.7667,128.7333,45,
78,korla,CN,41.7597,86.1469,26,
84,gurupi,BR,-11.7292,-49.0686,31,
96,luena,AO,-11.7833,19.9167,34,
100,whitehorse,CA,60.7161,-135.0538,37,
103,najafabad,IR,32.6344,51.3668,15,
110,grantsville,US,40.5999,-112.4644,23,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [27]:
# Set parameters to search for a hotel
radius = 15000  #Equates to 15 kilometers within the radius 
params = {
    "radius": radius,
    "type": "Hotel",
    "sort": "distance",
    "apiKey": geoapify_key
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lat = row['Lat']
    lng = row['Lng']
    
    # Add filter and bias parameters with the current city's latitude and longitude to the params dictionary
    params["filter"] = lat
    params["bias"] = lng
    
    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"


    # Make and API request using the params dictionaty
    name_address = requests.get(base_url, params=params)
    
    # Convert the API response to JSON format
    name_address_1 = name_address.json()
    
    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address_1["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"
        
    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
glen innes - nearest hotel: No hotel found
anaconda - nearest hotel: No hotel found
tindouf - nearest hotel: No hotel found
kununurra - nearest hotel: No hotel found
korla - nearest hotel: No hotel found
gurupi - nearest hotel: No hotel found
luena - nearest hotel: No hotel found
whitehorse - nearest hotel: No hotel found
najafabad - nearest hotel: No hotel found
grantsville - nearest hotel: No hotel found
loreto - nearest hotel: No hotel found
qom - nearest hotel: No hotel found
swift current - nearest hotel: No hotel found
mandalgovi - nearest hotel: No hotel found
hwange - nearest hotel: No hotel found
aioun - nearest hotel: No hotel found
darwin - nearest hotel: No hotel found
andkhoy - nearest hotel: No hotel found
thames - nearest hotel: No hotel found
hotan - nearest hotel: No hotel found
airway heights - nearest hotel: No hotel found
fernley - nearest hotel: No hotel found
sultanah - nearest hotel: No hotel found
qorveh - nearest hotel: No hotel found
roma

,City,Country,Lat,Lng,Humidity,Hotel name,Hotel Name
21,glen innes,AU,-29.7333,151.7333,47,,No hotel found
22,anaconda,US,46.1285,-112.9423,33,,No hotel found
43,tindouf,DZ,27.6711,-8.1474,15,,No hotel found
66,kununurra,AU,-15.7667,128.7333,45,,No hotel found
78,korla,CN,41.7597,86.1469,26,,No hotel found
...,...,...,...,...,...,...,...
557,tshikapa,CD,-6.4167,20.8000,31,,No hotel found
558,baharly,TM,38.4362,57.4316,33,,No hotel found
560,pontes e lacerda,BR,-15.2261,-59.3353,35,,No hotel found
561,tomelloso,ES,39.1522,-3.0243,42,,No hotel found


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [28]:
%%capture --no-display

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    x= 'Lng', y='Lat',
    geo= True, tiles='OSM',
    size='Humidity',
    color='City',
    cmap='Category10',
    legend='bottom_right',
    alpha=0.5,
    hover_cols=['City','Humidity','Country', 'Hotel Name']
)

#Display the map plot
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Country)